# ABW v0.1.2 — 다른 CPU 검증

런타임은 **CPU**를 사용합니다. `런타임 → 모두 실행`을 선택하세요.
공식 데이터 약 62 MB와 검증된 Boost 아카이브를 내려받습니다.
전체 새 실제 입력 9개, 기존과 동일한 입력/정답 해시, 27개 프로세스·2,544회 정렬을 검증합니다.
v0.1.1/v0.1.2/pdqsort/Spreadsort/CAFS u32·i32/Counting을 비교합니다.
알고리즘은 변경하지 않습니다. 입력 로딩·복원·정확성 검사는 타이머 밖에 있습니다.

Colab CPU/OS/GCC 환경과 로컬 Windows/Clang 환경의 차이를 기록합니다.
다른 CPU 결과를 로컬 결과와 섞어 평균내지 않습니다. 무료 공유 런타임의 변동이 남습니다.
원시 데이터의 라이선스/출처, 사전 고정한 열과 기존 원시 측정값은 번들에 포함돼 있습니다.
실행 후 마지막 셀이 결과 ZIP을 내려받습니다. 오류가 있으면 셀의 오류를 그대로 보존하세요.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy==2.3.5', 'pyarrow==26.0.1'])


In [ ]:
from pathlib import Path
import base64, zipfile, io, subprocess, sys, datetime
root=Path('/content')/('abw_v012_cpu_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
root.mkdir(); reference=root/'reference'; reference.mkdir()
bundle=base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(bundle)) as z: z.extractall(reference)
(root/'colab_run.py').write_text('from pathlib import Path\nimport json,hashlib,urllib.request,tarfile,subprocess,shutil,os,platform,random,datetime,statistics,math,zipfile\nimport numpy as np\nroot=Path.cwd();ref=root/\'reference\';w=root/\'work/abw_new_real_v012\';o=root/\'results\';w.mkdir(parents=True,exist_ok=True);o.mkdir(exist_ok=True)\nsha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()\nfor name,h in json.loads((ref/\'package_manifest.json\').read_text()).items():assert sha(ref/name)==h,name\nfrozen=json.loads((ref/\'input_manifest.json\').read_text());sources=frozen[\'sources\'];d=w/\'downloads\';d.mkdir(exist_ok=True)\nfor s in sources:\n p=d/s[\'file\'];print(\'Download\',s[\'id\'],flush=True)\n if not p.exists():\n  with urllib.request.urlopen(s[\'url\'],timeout=120) as f,p.open(\'wb\') as out:shutil.copyfileobj(f,out)\n assert sha(p)==s[\'sha256\'],s[\'id\']\nfor name in [\'sources.json\',\'design.json\']:shutil.copyfile(ref/name,w/name)\n# Use the exact original extraction logic, stopping before its Windows build.\nextract=(ref/\'prepare.py\').read_text().split(\'# Frozen headers:\')[0]\nextract=extract.replace("Path(\'outputs/abw_new_real_v012\')","Path(\'results\')")\nexec(compile(extract,\'original_exact_extraction\',\'exec\'))\nfresh=json.loads((w/\'input_manifest.json\').read_text())\nassert {c[\'name\']:c[\'sha256\'] for c in fresh[\'cases\']}=={c[\'name\']:c[\'sha256\'] for c in frozen[\'cases\']}\nassert {c[\'name\']:c[\'expected_sha256\'] for c in fresh[\'cases\']}=={c[\'name\']:c[\'expected_sha256\'] for c in frozen[\'cases\']}\ncases=frozen[\'cases\'];(o/\'input_manifest.json\').write_text(json.dumps(frozen,indent=2))\ncpuinfo=Path(\'/proc/cpuinfo\').read_text();model=next(l.split(\':\',1)[1].strip() for l in cpuinfo.splitlines() if l.startswith(\'model name\'));flags=next(l for l in cpuinfo.splitlines() if l.startswith(\'flags\')).split()\nassert \'avx2\' in flags and \'bmi1\' in flags,\'AVX2/BMI1 required for existing comparison implementations\'\nallowed=sorted(os.sched_getaffinity(0));os.sched_setaffinity(0,{allowed[0]})\nboost=root/\'boost_1_90_0\';archive=root/\'boost.tar.gz\'\nif not boost.exists():\n print(\'Download verified Boost 1.90.0\',flush=True)\n with urllib.request.urlopen(\'https://archives.boost.io/release/1.90.0/source/boost_1_90_0.tar.gz\',timeout=120) as f,archive.open(\'wb\') as out:shutil.copyfileobj(f,out)\n assert sha(archive)==\'5e93d582aff26868d581a52ae78c7d8edf3f3064742c6e77901a1f18a437eea9\'\n with tarfile.open(archive) as tf:tf.extractall(root,members=[m for m in tf.getmembers() if m.name.startswith(\'boost_1_90_0/boost/\')],filter=\'data\')\nshutil.copytree(ref/\'include\',w/\'include\',dirs_exist_ok=True);shutil.copyfile(ref/\'v011.hpp\',w/\'v011.hpp\')\nsrc=(ref/\'benchmark.cpp\').read_text().replace(\'#include <windows.h>\',\'#include <sched.h>\')\nold=\'DWORD_PTR p,s;if(!GetProcessAffinityMask(GetCurrentProcess(),&p,&s)||!SetProcessAffinityMask(GetCurrentProcess(),p&(~p+1)))throw std::runtime_error("affinity");\'\nnew=\'cpu_set_t allowed;CPU_ZERO(&allowed);if(sched_getaffinity(0,sizeof(allowed),&allowed))throw std::runtime_error("affinity");int cpu=0;while(cpu<CPU_SETSIZE&&!CPU_ISSET(cpu,&allowed))++cpu;cpu_set_t selected;CPU_ZERO(&selected);CPU_SET(cpu,&selected);if(sched_setaffinity(0,sizeof(selected),&selected))throw std::runtime_error("affinity set");\'\nassert old in src;src=src.replace(old,new);(w/\'benchmark_linux.cpp\').write_text(src)\ncompiler=shutil.which(\'g++\');assert compiler\ncmd=[compiler,\'-std=c++20\',\'-O3\',\'-DNDEBUG\',\'-mavx2\',\'-mbmi\',\'-pthread\',\'-I\'+str(w/\'include\'),\'-I\'+str(boost),\'-I\'+str(ref/\'vendor\'),str(w/\'benchmark_linux.cpp\'),\'-o\',str(w/\'benchmark\')]\nq=subprocess.run(cmd,capture_output=True,text=True);(o/\'build.json\').write_text(json.dumps(dict(command=cmd,returncode=q.returncode,stdout=q.stdout,stderr=q.stderr),indent=2));assert q.returncode==0,q.stderr\nenv=dict(cpu_model=model,cpuinfo=cpuinfo,platform=platform.platform(),python=platform.python_version(),numpy=np.__version__,pyarrow=__import__(\'pyarrow\').__version__,compiler=subprocess.check_output([compiler,\'--version\'],text=True),allowed_cpus=allowed,selected_cpu=allowed[0],meminfo=Path(\'/proc/meminfo\').read_text(),utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),same_model_as_local=\'i5-14500\' in model,virtualized_host=True,scope=\'Different Colab runtime CPU/OS/compiler; compare algorithms within each machine, no attribution solely to microarchitecture\',public_header_sha256=sha(w/\'include/abw/abw.hpp\'),reference_public_header_sha256=sha(ref/\'include/abw/abw.hpp\'))\n(o/\'environment.json\').write_text(json.dumps(env,indent=2));print(\'CPU\',model,\'compiler\',env[\'compiler\'].splitlines()[0],flush=True)\nlocal=[json.loads(l) for l in (ref/\'session0.jsonl\').read_text().splitlines()];route={c[\'name\']:c[\'result\'] for c in local};total=0\nfor session in range(3):\n jobs=cases.copy();random.Random(527813+session).shuffle(jobs)\n with (o/f\'session{session}.jsonl\').open(\'x\') as f:\n  for i,c in enumerate(jobs):\n   q=subprocess.run([str(w/\'benchmark\'),str(w/\'inputs\'/(c[\'name\']+\'.bin\')),str(w/\'inputs\'/(c[\'name\']+\'.expected.bin\')),str(624581+session),str(int(c[\'cafs_i32_eligible\'])),str(int(c[\'counting_eligible\']))],capture_output=True,text=True,check=True);v=json.loads(q.stdout);assert v[\'verified\'] and len(v[\'runs\'])==(len(c[\'methods\'])-1)*16\n   for key in [\'baseline_route\',\'candidate_route\',\'candidate_bits\',\'candidate_shift\',\'direct_output\']:assert v[key]==route[c[\'name\']][key],(c[\'name\'],key)\n   total+=len(v[\'runs\']);f.write(json.dumps(dict(name=c[\'name\'],group=c[\'group\'],n=c[\'n\'],session=session,result=v))+\'\\n\');f.flush();print(session+1,i+1,\'/9\',c[\'name\'],flush=True)\nfor c in cases:\n assert sha(w/\'inputs\'/(c[\'name\']+\'.bin\'))==c[\'sha256\'];assert sha(w/\'inputs\'/(c[\'name\']+\'.expected.bin\'))==c[\'expected_sha256\']\n# Same estimator and CSV construction, without the historical Windows report text.\nanalysis=(ref/\'analyze.py\').read_text().split("text=\'\'\'")[0].replace("Path(\'outputs/abw_new_real_v012\')","Path(\'results\')")\nexec(compile(analysis,\'same_estimator\',\'exec\'))\nassert total==2544\n(o/\'validation.json\').write_text(json.dumps(dict(correctness=True,processes=27,timed_sorts=total,input_hashes_match=True,release_header_unchanged=env[\'public_header_sha256\']==env[\'reference_public_header_sha256\'],all_route_fields_match=True,cpu_model=model,same_cpu_model_as_local=env[\'same_model_as_local\'],github_unchanged=True),indent=2))\nfor name in [\'benchmark_linux.cpp\',\'v011.hpp\']:shutil.copyfile(w/name,o/name)\nshutil.copyfile(w/\'include/abw/abw.hpp\',o/\'v012.hpp\')\nreport=f"# ABW v0.1.2 Colab CPU validation\\n\\nCPU: {model}\\n\\nCompiler: {env[\'compiler\'].splitlines()[0]}\\n\\n27 processes, 2,544 sorts, every full-array check passed. All source input and expected hashes and routes match the Windows run. This is a hosted/virtualized runtime with a different OS/compiler; no hardware-only causal claim. No policy changes.\\n\\n| Method | N | GM speedup vs pdq | v0.1.2/method time |\\n|---|---:|---:|---:|\\n"\nfor name,v in metrics.items():report+=f"| {name} | {v[\'eligible\']} | {v[\'gm_speedup_pdq\']:.4f} | {v[\'gm_v012_over_method\']:.4f} |\\n"\nreport+=\'\\n| Input | v0.1.2 ms | v0.1.1 ms | pdq ms | CAFS u32 ms | CAFS i32 ms | v0.1.2/v0.1.1 | Direct |\\n|---|---:|---:|---:|---:|---:|---:|---|\\n\'\nfor x in rows:report+=f"| {x[\'name\']} | {x[\'v012\']:.5f} | {x[\'v011\']:.5f} | {x[\'pdq\']:.5f} | {x[\'CAFS_u32\']:.5f} | {x[\'CAFS_i32\']:.5f} | {x[\'v012_over_v011\']:.4f} | {x[\'direct\']} |\\n"\n(o/\'report.md\').write_text(report);print(report,flush=True)\nhashes={p.name:sha(p) for p in o.iterdir() if p.is_file()};(o/\'file_hashes.json\').write_text(json.dumps(hashes,indent=2))\nout=root/\'abw_v012_colab_cpu_results.zip\'\nwith zipfile.ZipFile(out,\'w\',zipfile.ZIP_DEFLATED) as z:\n for p in o.iterdir():\n  if p.is_file():z.write(p,p.name)\nprint(\'RESULT_ZIP\',out,flush=True)\n')
print('Working directory:', root)
subprocess.run([sys.executable, str(root/'colab_run.py')], cwd=root, check=True)


In [ ]:
from google.colab import files
files.download(str(root/'abw_v012_colab_cpu_results.zip'))
